# Parking Birmingham — Baseline and Regression Modeling


## 1. Introduction

This notebook evaluates a simple occupancy persistence baseline and two regression models using the leakage-safe features and chronological splits created in Notebook 2.

### Objectives

1. Load validated parking observations and construct forecasting features.
2. Preserve the chronological training, validation, and test splits.
3. Establish a persistence baseline using current occupancy.
4. Train Linear Regression and Random Forest models.
5. Compare validation MAE and RMSE.
6. Save reproducible model and metrics artifacts.

### Temporal Evaluation Strategy

The test split remains unused during model development. Model selection and hyperparameter choices are based only on the validation split, while the test split is reserved for a final evaluation conducted after the models are selected.

## 2. Environment and Data Preparation

The notebook loads the original dataset, applies the reusable validation function, creates historical features and forecast targets, and reproduces the chronological split. The project root is detected whether the notebook is run from the repository root or the `notebooks` directory.

In [6]:
import json
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

from src.data_validation import validate_parking_data
from src.feature_engineering import (
    create_historical_features,
    create_forecast_targets,
    chronological_split,
)

raw_df = pd.read_csv(project_root / "data" / "dataset.csv")
validated_df = validate_parking_data(raw_df)
historical_df = create_historical_features(validated_df)
forecast_df = create_forecast_targets(historical_df)
train_df, validation_df, test_df = chronological_split(forecast_df)

print("Original observations:", len(raw_df))
print("Validated observations:", len(validated_df))
print("Valid forecasting examples:", len(forecast_df))
print("Training examples:", len(train_df))
print("Validation examples:", len(validation_df))
print("Held-out test examples:", len(test_df))

Original observations: 35717
Validated observations: 35116
Valid forecasting examples: 31327
Training examples: 22467
Validation examples: 4143
Held-out test examples: 4666


## 3. Baseline and Model Training

The persistence baseline predicts that occupancy will remain equal to the current occupancy. Linear Regression estimates a linear relationship between occupancy and the shared feature set. Random Forest captures nonlinear relationships and interactions while using only data available at prediction time.

Because `SystemCodeNumber` is categorical, it is one-hot encoded. Numeric features are median-imputed. All preprocessing is fitted using training data only.

In [7]:
feature_columns = [
    "SystemCodeNumber",
    "Capacity",
    "Occupancy",
    "hour",
    "day_of_week",
    "is_weekend",
    "month",
    "occupancy_lag_1",
    "occupancy_lag_2",
    "occupancy_change_1",
    "occupancy_rolling_mean_3",
]
target_column = "TargetOccupancy"

train_model_df = train_df.dropna(subset=feature_columns + [target_column]).copy()
validation_model_df = validation_df.dropna(subset=feature_columns + [target_column]).copy()

X_train = train_model_df[feature_columns]
y_train = train_model_df[target_column]
X_validation = validation_model_df[feature_columns]
y_validation = validation_model_df[target_column]

categorical_features = ["SystemCodeNumber"]
numeric_features = [column for column in feature_columns if column not in categorical_features]

preprocessing = ColumnTransformer([
    (
        "categorical",
        Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore")),
        ]),
        categorical_features,
    ),
    (
        "numeric",
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
        ]),
        numeric_features,
    ),
])

linear_model = Pipeline([
    ("preprocessing", preprocessing),
    ("model", LinearRegression()),
])

random_forest_model = Pipeline([
    ("preprocessing", preprocessing),
    (
        "model",
        RandomForestRegressor(
            n_estimators=200,
            random_state=42,
            n_jobs=-1,
            min_samples_leaf=2,
        ),
    ),
])

linear_model.fit(X_train, y_train)
random_forest_model.fit(X_train, y_train)

baseline_predictions = X_validation["Occupancy"].to_numpy()
linear_predictions = linear_model.predict(X_validation)
random_forest_predictions = random_forest_model.predict(X_validation)

# Predictions must represent physically plausible occupancy values.
baseline_predictions = np.clip(baseline_predictions, 0, X_validation["Capacity"])
linear_predictions = np.clip(linear_predictions, 0, X_validation["Capacity"])
random_forest_predictions = np.clip(
    random_forest_predictions,
    0,
    X_validation["Capacity"],
)

print("Models trained successfully.")
print("Training examples:", len(X_train))
print("Validation examples:", len(X_validation))

Models trained successfully.
Training examples: 18627
Validation examples: 3389


## 4. Validation Results

Mean Absolute Error (MAE) measures the average absolute difference between predicted and actual occupied spaces. Root Mean Squared Error (RMSE) penalizes larger prediction errors more heavily.

The test split is not used in this comparison.

In [8]:
def evaluate_model(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    return {"MAE": float(mae), "RMSE": float(rmse)}


metrics = {
    "Persistence Baseline": evaluate_model(
        y_validation,
        baseline_predictions,
    ),
    "Linear Regression": evaluate_model(
        y_validation,
        linear_predictions,
    ),
    "Random Forest": evaluate_model(
        y_validation,
        random_forest_predictions,
    ),
}

metrics_df = pd.DataFrame(metrics).T
metrics_df.index.name = "Model"
metrics_df.sort_values("RMSE")

,MAE,RMSE
Model,,
Random Forest,18.775141,33.672739
Linear Regression,20.013103,33.674048
Persistence Baseline,54.100620,93.675742


## 5. Save Reproducible Artifacts

The final Linear Regression and Random Forest pipelines are saved as joblib files. Validation metrics are saved separately so the comparison can be reproduced without rerunning the models.

In [9]:
artifact_dir = project_root / "artifacts"
artifact_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(linear_model, artifact_dir / "linear_regression.joblib")
joblib.dump(random_forest_model, artifact_dir / "random_forest.joblib")

with (artifact_dir / "validation_metrics.json").open("w") as file:
    json.dump(metrics, file, indent=2)

print("Saved Linear Regression pipeline:", artifact_dir / "linear_regression.joblib")
print("Saved Random Forest pipeline:", artifact_dir / "random_forest.joblib")
print("Saved validation metrics:", artifact_dir / "validation_metrics.json")

Saved Linear Regression pipeline: /Users/monicapara/Desktop/parking-predictability/artifacts/linear_regression.joblib
Saved Random Forest pipeline: /Users/monicapara/Desktop/parking-predictability/artifacts/random_forest.joblib
Saved validation metrics: /Users/monicapara/Desktop/parking-predictability/artifacts/validation_metrics.json
